In [1]:
import dspy
import os
from dotenv import load_dotenv
from typing import Literal

load_dotenv()

ANTHROPIC_KEY = os.getenv("ANTHROPIC_KEY")
OPENAI_KEY = os.getenv("OPENAI_KEY")

In [2]:
teacher_model = dspy.LM("anthropic/claude-opus-4-6", api_key=ANTHROPIC_KEY)
student_model = dspy.LM("anthropic/claude-haiku-4-5-20251001", api_key=ANTHROPIC_KEY)

## sentiment classifier using bootstrap optimizer

In [3]:
headlines = [
    {"headline": "Apple reports record Q4 revenue, beating analyst expectations by 15%", "sentiment": "bullish", "impact": "high"},
    {"headline": "Fed signals potential rate hike in upcoming meeting amid inflation concerns", "sentiment": "bearish", "impact": "high"},
    {"headline": "Tesla recalls 50,000 vehicles over minor software glitch in dashboard display", "sentiment": "bearish", "impact": "low"},
    {"headline": "Google announces routine update to search algorithm", "sentiment": "neutral", "impact": "low"},
    {"headline": "Amazon expands same-day delivery to 20 new cities", "sentiment": "bullish", "impact": "medium"},
    {"headline": "Microsoft lays off 5% of workforce in cloud division restructuring", "sentiment": "bearish", "impact": "medium"},
    {"headline": "S&P 500 closes flat after mixed earnings reports from tech sector", "sentiment": "neutral", "impact": "low"},
    {"headline": "Nvidia secures $10B government contract for AI infrastructure", "sentiment": "bullish", "impact": "high"},
    {"headline": "Oil prices surge 8% after OPEC announces unexpected production cuts", "sentiment": "bullish", "impact": "high"},
    {"headline": "Meta faces new antitrust investigation in European Union", "sentiment": "bearish", "impact": "medium"},
    {"headline": "Warren Buffett increases Berkshire's cash reserves to all-time high", "sentiment": "neutral", "impact": "medium"},
    {"headline": "Startup backed by Andreessen Horowitz files for bankruptcy after 18 months", "sentiment": "bearish", "impact": "low"},
]

In [4]:
## Create test/train set

dspy.configure(lm=student_model, track_usage=True, adapter=dspy.JSONAdapter())
full_data = [dspy.Example(headline).with_inputs("headline") for headline in headlines]

train_set = full_data[0:5]
test_set = full_data[5:]

#Create a signature 
class HeadlineSentiment(dspy.Signature):
    """Classify the new headlines to deduce 2 things.
       a)Sentiment — Investor sentiment and market outlook based on the headline.  
       b)Impact — Expected magnitude of market or stock price movement."""

    headline:str = dspy.InputField(desc="News headline describing anevent that might influence investor decisions")
    
    sentiment:Literal["bullish", "bearish", "neutral"] = dspy.OutputField(desc="Classify into one of the \
    provided category. See the definitions of catgeory - Bullish — positive for stock price or market. \
    Good earnings, new revenue, expansion.\
    Bearish — negative for stock price or market. Layoffs, lawsuits, rate hikes.\
    Neutral — no clear directional signal. Flat results, routine news, mixed signals.")
    
    impact:Literal["high", "medium", "low"] = dspy.OutputField(desc="Classify into one of the \
    provided category. See the definitions of category - High — earnings beats/misses,\
    Fed policy, major contracts. Traders act on these immediately.\
    Medium — layoffs, expansion news, regulatory investigations. Notable but not market-shaking.\
    Low — routine updates, minor recalls, flat trading days. Noise.")


In [5]:
def sentiment_evaluator(data_instance, prediction, trace=None):

    return (data_instance.sentiment.lower() == prediction.sentiment.lower()) and (
        data_instance.impact.lower() == prediction.impact.lower())

def invoke_sentiment_evaluator(module, result_filename):

    sentiment_eval = dspy.Evaluate(
        devset=test_set,
        metric=sentiment_evaluator,
        display_progress=True,
        save_as_json=result_filename
    )

    score = sentiment_eval(module)
    return score




In [6]:

plain_cot_classifier = dspy.ChainOfThought(signature=HeadlineSentiment)
invoke_sentiment_evaluator(plain_cot_classifier, "./exploration_evaluation_results/headlines_sentiment_unoptimized.json")

##~57% accuracy

Average Metric: 4.00 / 7 (57.1%): 100%|██████████████████████| 7/7 [00:00<00:00, 15.80it/s]

2026/03/18 16:19:25 INFO dspy.evaluate.evaluate: Average Metric: 4 / 7 (57.1%)


EvaluationResult(score=57.14, results=<list of 7 results>)

In [7]:
dspy.inspect_history(n=1)





[2026-03-18T16:19:25.523000]

System message:

Your input fields are:
1. `headline` (str): News headline describing anevent that might influence investor decisions
Your output fields are:
1. `reasoning` (str): 
2. `sentiment` (Literal['bullish', 'bearish', 'neutral']): Classify into one of the     provided category. See the definitions of catgeory - Bullish — positive for stock price or market.     Good earnings, new revenue, expansion.    Bearish — negative for stock price or market. Layoffs, lawsuits, rate hikes.    Neutral — no clear directional signal. Flat results, routine news, mixed signals.
3. `impact` (Literal['high', 'medium', 'low']): Classify into one of the     provided category. See the definitions of category - High — earnings beats/misses,    Fed policy, major contracts. Traders act on these immediately.    Medium — layoffs, expansion news, regulatory investigations. Notable but not market-shaking.    Low — routine updates, minor recalls, flat trading days. Noise.
A

In [8]:
##Creating an optimizer - no teacher model

optimizer = dspy.BootstrapFewShot(
    metric=sentiment_evaluator,
    teacher_settings=None, #Test it in another run to isolate the effct of optimizer alone
    max_bootstrapped_demos=3,
    max_labeled_demos=4,
    max_rounds=10    
)

optimized_cot_classifier = optimizer.compile(plain_cot_classifier, trainset=train_set)

invoke_sentiment_evaluator(optimized_cot_classifier , "./exploration_evaluation_results/headlines_sentiment_optimized.json")

## A bump in accuray to 71% for the max_rounds=10

 60%|█████████████████████████████████▌                      | 3/5 [00:00<00:00, 19.77it/s]

Bootstrapped 3 full traces after 3 examples for up to 10 rounds, amounting to 4 attempts.


Average Metric: 5.00 / 7 (71.4%): 100%|█████████████████████| 7/7 [00:00<00:00, 255.08it/s]

2026/03/18 16:19:25 INFO dspy.evaluate.evaluate: Average Metric: 5 / 7 (71.4%)


EvaluationResult(score=71.43, results=<list of 7 results>)

In [9]:
dspy.inspect_history(n=1)





[2026-03-18T16:19:25.768687]

System message:

Your input fields are:
1. `headline` (str): News headline describing anevent that might influence investor decisions
Your output fields are:
1. `reasoning` (str): 
2. `sentiment` (Literal['bullish', 'bearish', 'neutral']): Classify into one of the     provided category. See the definitions of catgeory - Bullish — positive for stock price or market.     Good earnings, new revenue, expansion.    Bearish — negative for stock price or market. Layoffs, lawsuits, rate hikes.    Neutral — no clear directional signal. Flat results, routine news, mixed signals.
3. `impact` (Literal['high', 'medium', 'low']): Classify into one of the     provided category. See the definitions of category - High — earnings beats/misses,    Fed policy, major contracts. Traders act on these immediately.    Medium — layoffs, expansion news, regulatory investigations. Notable but not market-shaking.    Low — routine updates, minor recalls, flat trading days. Noise.
A

In [10]:
##next run  -  Use teacher, a more advanced model to direct sample generation
#Optimize the same student

optimizer_senti_teacher = dspy.BootstrapFewShot(
    metric=sentiment_evaluator,
    teacher_settings=dict(lm=teacher_model),
    max_bootstrapped_demos=3,
    max_labeled_demos=4,
    max_rounds=10  
)

optimized_cot_teacher = optimizer_senti_teacher.compile(plain_cot_classifier, trainset=train_set)

invoke_sentiment_evaluator(optimized_cot_teacher , "./exploration_evaluation_results/headlines_sentiment_optimized_teacher.json")
##With max_rounds=10; The accuracy was ~71%

## Teacher model didnt make much of a difference here

 60%|█████████████████████████████████▌                      | 3/5 [00:00<00:00, 19.64it/s]


Bootstrapped 3 full traces after 3 examples for up to 10 rounds, amounting to 3 attempts.
Average Metric: 5.00 / 7 (71.4%): 100%|█████████████████████| 7/7 [00:00<00:00, 187.72it/s]

2026/03/18 16:19:26 INFO dspy.evaluate.evaluate: Average Metric: 5 / 7 (71.4%)


EvaluationResult(score=71.43, results=<list of 7 results>)

In [11]:
dspy.inspect_history(n=1)





[2026-03-18T16:19:26.009835]

System message:

Your input fields are:
1. `headline` (str): News headline describing anevent that might influence investor decisions
Your output fields are:
1. `reasoning` (str): 
2. `sentiment` (Literal['bullish', 'bearish', 'neutral']): Classify into one of the     provided category. See the definitions of catgeory - Bullish — positive for stock price or market.     Good earnings, new revenue, expansion.    Bearish — negative for stock price or market. Layoffs, lawsuits, rate hikes.    Neutral — no clear directional signal. Flat results, routine news, mixed signals.
3. `impact` (Literal['high', 'medium', 'low']): Classify into one of the     provided category. See the definitions of category - High — earnings beats/misses,    Fed policy, major contracts. Traders act on these immediately.    Medium — layoffs, expansion news, regulatory investigations. Notable but not market-shaking.    Low — routine updates, minor recalls, flat trading days. Noise.
A